In [ ]:
import duckdb


duckdb.sql(""" 
    CREATE OR REPLACE TABLE cities AS
    WITH my_cities AS( 
        SELECT parse_filename(filename, true) AS city,
        COUNT (*) AS active_entire_homes
        FROM read_csv_auto('../data/*.csv', filename = true, union_by_name = true)
        WHERE parse_filename(filename, true) NOT IN ('geneva','zurich','vaud')
        AND number_of_reviews_ltm >0
        AND room_type='Entire home/apt'
        GROUP BY parse_filename(filename, true)
        HAVING COUNT(*) >= 500),
    numbeo AS (
        SELECT city AS numbeo_city,
        replace(lower(split_part(city, ',', 1)), ' ', '-') AS join_key,
        price_sqm
        FROM read_csv_auto('../data/reference/numbeo_centre.csv')),
    city_lookup AS (
        SELECT * FROM read_csv_auto('../data/reference/city_lookup.csv'))
    
    SELECT m.city,
        m.active_entire_homes,
        n.numbeo_city,
        n.price_sqm
    FROM my_cities m
    LEFT JOIN city_lookup c ON m.city=c.city
    LEFT JOIN numbeo n ON n.numbeo_city=c.numbeo_city
                        OR (c.numbeo_city IS NULL AND n.join_key=m.city)
    ORDER BY n.price_sqm NULLS FIRST
    
    """).df()


duckdb.sql("""
CREATE OR REPLACE TABLE city_metrics AS
WITH listing_nights AS (
    SELECT
        parse_filename(filename, true) AS city,
        price,
        LEAST((number_of_reviews_ltm / 0.5) * GREATEST(3, minimum_nights), 255) AS nights_booked,
        (number_of_reviews_ltm / 0.5) * GREATEST(3, minimum_nights) AS nights_uncapped
    FROM read_csv_auto('../data/*.csv', filename = true, union_by_name = true)
    WHERE room_type = 'Entire home/apt'
      AND number_of_reviews_ltm > 0
),
agg AS (
    SELECT
        city,
        MEDIAN(price) AS med_price_local,
        MEDIAN(nights_booked) AS med_nights,
        ROUND(100.0 * SUM(CASE WHEN nights_uncapped > 255 THEN 1 ELSE 0 END) / COUNT(*), 1) AS pct_capped
    FROM listing_nights
    GROUP BY city
),
cur AS (SELECT * FROM read_csv_auto('../data/reference/country_currency.csv')),
ex  AS (SELECT * FROM read_csv_auto('../data/reference/exchange_rates.csv')),
out AS (SELECT city AS numbeo_city, price_sqm AS price_sqm_out
        FROM read_csv_auto('../data/reference/numbeo_outside.csv')),
utl AS (SELECT city AS numbeo_city, price_utilities
        FROM read_csv_auto('../data/reference/numbeo_utilities.csv'))
SELECT
    c.city,
    split_part(c.numbeo_city, ',', 1) AS city_display,
    c.numbeo_city,
    ROUND(a.med_price_local / ex.usd_rate, 2) AS med_airbnb_price_usd,
    ROUND(a.med_nights, 0) AS med_nights,
    a.pct_capped,
    c.price_sqm AS price_sqm_centre,
    out.price_sqm_out,
    utl.price_utilities,
    ROUND(100.0 * (a.med_price_local / ex.usd_rate * a.med_nights) / (c.price_sqm * 60)
          - 100.0 * (12 * utl.price_utilities) / (85 * c.price_sqm) - 1.0, 2) AS roi_centre,
    ROUND(100.0 * (a.med_price_local / ex.usd_rate * a.med_nights) / (out.price_sqm_out * 60)
          - 100.0 * (12 * utl.price_utilities) / (85 * out.price_sqm_out) - 1.0, 2) AS roi_outside
FROM cities c
JOIN agg a   ON a.city = c.city
JOIN cur     ON cur.country = trim(split_part(c.numbeo_city, ',', -1))
JOIN ex      ON ex.currency = cur.currency
LEFT JOIN out ON out.numbeo_city = c.numbeo_city
LEFT JOIN utl ON utl.numbeo_city = c.numbeo_city
WHERE c.city <> 'new-york-city'
""")

AttributeError: 'NoneType' object has no attribute 'df'

In [3]:
duckdb.sql("""DESCRIBE SELECT * FROM read_csv_auto('../data/*.csv', filename = true, union_by_name = true)""")

┌────────────────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│          column_name           │ column_type │  null   │   key   │ default │  extra  │
│            varchar             │   varchar   │ varchar │ varchar │ varchar │ varchar │
├────────────────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ id                             │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ name                           │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ host_id                        │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ host_profile_id                │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ host_name                      │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ neighbourhood_group            │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ neighbourhood                  │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ latitude           

In [4]:
import os
os.getcwd()


'c:\\Users\\Ab\\Desktop\\airbnb-roi-analysis'

In [5]:
duckdb.sql("""
    SELECT 
        CASE
          WHEN number_of_reviews_ltm = 0 THEN '0'
          WHEN number_of_reviews_ltm <= 2 THEN '1-2'
          WHEN number_of_reviews_ltm <= 5 THEN '3-5'
            WHEN number_of_reviews_ltm <= 12 THEN '6-12'
            ELSE '13+'
        END AS band,
        COUNT (*) AS n
    FROM read_csv_auto('../data/*.csv', union_by_name = true)
    WHERE room_type = 'Entire home/apt'
    GROUP BY band
    ORDER BY band""")

┌─────────┬────────┐
│  band   │   n    │
│ varchar │ int64  │
├─────────┼────────┤
│ 0       │ 429520 │
│ 1-2     │ 197544 │
│ 13+     │ 370668 │
│ 3-5     │ 157154 │
│ 6-12    │ 191999 │
└─────────┴────────┘

In [6]:
duckdb.sql("""
SELECT
    parse_filename(filename, true) AS city,
    COUNT(*) AS active_entire_homes
FROM read_csv_auto('../data/*.csv', filename = true, union_by_name = true)
WHERE room_type = 'Entire home/apt'
  AND number_of_reviews_ltm > 0
  AND parse_filename(filename, true) NOT IN ('zurich', 'geneva', 'vaud')
GROUP BY parse_filename(filename, true)
HAVING COUNT(*) >= 500
ORDER BY active_entire_homes DESC
""").df()

,city,active_entire_homes
0,paris,37404
1,new-zealand,34736
2,london,33455
3,sao-paulo,30168
4,rio-de-janeiro,28193
...,...,...
108,ghent,824
109,the-hague,821
110,newark,788
111,rotterdam,691


In [9]:
duckdb.sql("""
SELECT
    parse_filename(filename, true) AS city,
    COUNT(*) AS active_entire_homes
FROM read_csv_auto('../data/*.csv', filename = true, union_by_name = true)
WHERE room_type = 'Entire home/apt'
  AND number_of_reviews_ltm > 0
  AND parse_filename(filename, true) NOT IN ('zurich', 'geneva', 'vaud')
GROUP BY parse_filename(filename, true)
HAVING COUNT(*) >= 500
ORDER BY active_entire_homes 
LIMIT 50
""").df()

,city,active_entire_homes
0,rochester,552
1,rotterdam,691
2,newark,788
3,the-hague,821
4,ghent,824
5,jersey-city,1078
6,winnipeg,1088
7,ottawa,1139
8,oakland,1166
9,santa-cruz-county,1212


In [11]:
duckdb.sql("""
SELECT neighbourhood, COUNT(*) AS n
FROM read_csv_auto('../data/los-angeles.csv')
GROUP BY neighbourhood ORDER BY n DESC LIMIT 50
""").df()

,neighbourhood,n
0,Long Beach,1856
1,Hollywood,1598
2,Venice,1544
3,West Hollywood,1282
4,Santa Monica,1229
5,Downtown,1090
6,Exposition Park,906
7,Pasadena,805
8,Beverly Hills,791
9,Glendale,722


In [13]:
import duckdb

duckdb.sql(""" DESCRIBE SELECT * FROM read_csv_auto('../data/reference/numbeo_centre.csv')""").df()

,column_name,column_type,null,key,default,extra
0,rank,BIGINT,YES,None,None,None
1,city,VARCHAR,YES,None,None,None
2,price_sqm,DOUBLE,YES,None,None,None


In [16]:
duckdb.sql(""" SELECT * FROM read_csv_auto('../data/reference/numbeo_centre.csv')""").df()

,rank,city,price_sqm
0,1,"Zug, Switzerland",31700.56
1,2,"Hong Kong, Hong Kong (China)",28216.99
2,3,"Zurich, Switzerland",27662.48
3,4,"Seoul, South Korea",27261.85
4,5,"Singapore, Singapore",23113.46
...,...,...,...
509,510,"Rajkot, India",732.83
510,511,"Dehradun, India",730.39
511,512,"Alexandria, Egypt",726.02
512,513,"Kabul, Afghanistan",681.65


In [18]:
duckdb.sql("""
WITH my_cities AS (
    SELECT
        parse_filename(filename, true) AS city,
        COUNT(*) AS active_entire_homes
    FROM read_csv_auto('../data/*.csv', filename = true, union_by_name = true)
    WHERE room_type = 'Entire home/apt'
      AND number_of_reviews_ltm > 0
      AND parse_filename(filename, true) NOT IN ('geneva', 'zurich', 'vaud')
    GROUP BY parse_filename(filename, true)
    HAVING COUNT(*) >= 500
),
numbeo AS (
    SELECT
        city AS numbeo_city,
        replace(lower(split_part(city, ',', 1)), ' ', '-') AS join_key,
        price_sqm
    FROM read_csv_auto('../data/reference/numbeo_centre.csv')
)
SELECT
    m.city,
    m.active_entire_homes,
    n.numbeo_city,
    n.price_sqm
FROM my_cities m
LEFT JOIN numbeo n ON m.city = n.join_key
ORDER BY n.price_sqm NULLS FIRST
LIMIT 50
""").df()

,city,active_entire_homes,numbeo_city,price_sqm
0,broward-county,10772,None,NaN
1,puglia,20105,None,NaN
2,pays-basque,7441,None,NaN
3,santa-cruz-county,1212,None,NaN
4,malta,7509,None,NaN
5,mornington-peninsula,3123,None,NaN
6,mallorca,9512,None,NaN
7,south-aegean,19987,None,NaN
8,sunshine-coast,5048,None,NaN
9,new-brunswick,3122,None,NaN


In [19]:
duckdb.sql("""
SELECT replace(lower(split_part(city, ',', 1)), ' ', '-') AS join_key,
       COUNT(*) AS n, string_agg(city, ' | ') AS matches
FROM read_csv_auto('../data/reference/numbeo_centre.csv')
GROUP BY 1 HAVING COUNT(*) > 1
""").df()

,join_key,n,matches
0,london,2,"London, United Kingdom | London, Canada"
1,vancouver,2,"Vancouver, Canada | Vancouver, WA, United States"
2,san-jose,2,"San Jose, CA, United States | San Jose, Costa ..."
3,birmingham,2,"Birmingham, United Kingdom | Birmingham, AL, U..."


In [20]:
duckdb.sql("""
    SELECT * FROM read_csv_auto('../data/reference/numbeo_centre.csv')
    WHERE city LIKE '%New York%' OR city LIKE '%Washington%' OR city LIKE '%Newark%'
   OR city LIKE '%Asheville%' OR city LIKE '%Venice%' OR city LIKE '%Sevill%'
   OR city LIKE '%Ghent%' OR city LIKE '%Gent%' OR city LIKE '%Hague%'
   OR city LIKE '%Las Vegas%' OR city LIKE '%Minneapolis%' OR city LIKE '%Manchester%'
   OR city LIKE '%Lauderdale%' OR city LIKE '%San Jose%' OR city LIKE '%San Mateo%'
   OR city LIKE '%Santa Cruz%' OR city LIKE '%Vancouver%'
   ORDER BY city""")

┌───────┬────────────────────────────────────┬───────────┐
│ rank  │                city                │ price_sqm │
│ int64 │              varchar               │  double   │
├───────┼────────────────────────────────────┼───────────┤
│   151 │ Fort Lauderdale, FL, United States │   5510.54 │
│   137 │ Gent, Belgium                      │   5826.41 │
│   172 │ Las Vegas, NV, United States       │   5208.26 │
│   183 │ Manchester, United Kingdom         │   5007.15 │
│   292 │ Minneapolis, MN, United States     │   3395.59 │
│    12 │ New York, NY, United States        │  18759.46 │
│    55 │ San Jose, CA, United States        │   8725.79 │
│   413 │ San Jose, Costa Rica               │   2281.02 │
│   481 │ Santa Cruz, Bolivia                │   1275.96 │
│   205 │ Seville (Sevilla), Spain           │   4599.82 │
│   121 │ The Hague (Den Haag), Netherlands  │   6406.35 │
│    52 │ Vancouver, Canada                  │   8985.48 │
│   294 │ Vancouver, WA, United States       │    3366.5

In [21]:
duckdb.sql("SELECT * FROM read_csv_auto('../data/reference/city_lookup.csv')").df()

,city,numbeo_city
0,new-york-city,"New York, NY, United States"
1,washington-dc,"Washington, DC, United States"
2,sevilla,"Seville (Sevilla), Spain"
3,ghent,"Gent, Belgium"
4,the-hague,"The Hague (Den Haag), Netherlands"
5,clark-county-nv,"Las Vegas, NV, United States"
6,twin-cities-msa,"Minneapolis, MN, United States"
7,greater-manchester,"Manchester, United Kingdom"
8,broward-county,"Fort Lauderdale, FL, United States"
9,santa-clara-county,"San Jose, CA, United States"


In [42]:
duckdb.sql("""
    WITH my_cities AS( 
        SELECT parse_filename(filename, true) AS city,
        COUNT (*) AS active_entire_homes
        FROM read_csv_auto('../data/*.csv', filename = true, union_by_name = true)
        WHERE parse_filename(filename, true) NOT IN ('geneva','zurich','vaud')
        AND number_of_reviews_ltm >0
        AND room_type='Entire home/apt'
        GROUP BY parse_filename(filename, true)
        HAVING COUNT(*) >= 500),
    numbeo AS (
        SELECT city AS numbeo_city,
        replace(lower(split_part(city, ',', 1)), ' ', '-') AS join_key,
        price_sqm
        FROM read_csv_auto('../data/reference/numbeo_centre.csv')),
    city_lookup AS (
        SELECT * FROM read_csv_auto('../data/reference/city_lookup.csv'))
    
    SELECT m.city,
        m.active_entire_homes,
        n.numbeo_city,
        n.price_sqm
    FROM my_cities m
    LEFT JOIN city_lookup c ON m.city=c.city
    LEFT JOIN numbeo n ON n.numbeo_city=c.numbeo_city
                        OR (c.numbeo_city IS NULL AND n.join_key=m.city)
    ORDER BY n.price_sqm NULLS FIRST
    
    """).df()


,city,active_entire_homes,numbeo_city,price_sqm
0,san-mateo-county,1583,None,NaN
1,euskadi,3890,None,NaN
2,rhode-island,3642,None,NaN
3,puglia,20105,None,NaN
4,mornington-peninsula,3123,None,NaN
...,...,...,...,...
108,vienna,7578,"Vienna, Austria",14565.33
109,paris,37404,"Paris, France",15008.74
110,new-york-city,5763,"New York, NY, United States",18759.46
111,london,33455,"London, United Kingdom",20204.52


In [44]:
duckdb.sql("""
    SELECT * FROM cities""")

┌──────────────────────┬─────────────────────┬──────────────────────────────┬───────────┐
│         city         │ active_entire_homes │         numbeo_city          │ price_sqm │
│       varchar        │        int64        │           varchar            │  double   │
├──────────────────────┼─────────────────────┼──────────────────────────────┼───────────┤
│ san-mateo-county     │                1583 │ NULL                         │      NULL │
│ euskadi              │                3890 │ NULL                         │      NULL │
│ mornington-peninsula │                3123 │ NULL                         │      NULL │
│ belize               │                1708 │ NULL                         │      NULL │
│ santa-cruz-county    │                1212 │ NULL                         │      NULL │
│ newark               │                 788 │ NULL                         │      NULL │
│ pays-basque          │                7441 │ NULL                         │      NULL │
│ menorca 

In [47]:
duckdb.sql("""
    SELECT 
        COUNT (*) AS n,
        SUM(CASE WHEN price_sqm IS NULL THEN 1 ELSE 0 END) AS null_price_sqm,
        SUM(CASE WHEN price_sqm IS NOT NULL THEN 1 ELSE 0 END) AS n_price_sqm
    FROM cities""")
    

┌───────┬────────────────┬─────────────┐
│   n   │ null_price_sqm │ n_price_sqm │
│ int64 │     int128     │   int128    │
├───────┼────────────────┼─────────────┤
│   113 │             29 │          84 │
└───────┴────────────────┴─────────────┘

In [48]:
duckdb.sql(""" DELETE FROM cities WHERE price_sqm IS NULL""")

In [49]:
duckdb.sql("""
    SELECT * FROM cities""").df()

,city,active_entire_homes,numbeo_city,price_sqm
0,nairobi,5749,"Nairobi, Kenya",1436.47
1,rochester,552,"Rochester, NY, United States",1462.56
2,winnipeg,1088,"Winnipeg, Canada",2341.75
3,cape-town,16317,"Cape Town, South Africa",2388.51
4,buenos-aires,20895,"Buenos Aires, Argentina",2644.00
...,...,...,...,...
79,vienna,7578,"Vienna, Austria",14565.33
80,paris,37404,"Paris, France",15008.74
81,new-york-city,5763,"New York, NY, United States",18759.46
82,london,33455,"London, United Kingdom",20204.52


In [52]:
duckdb.sql("""
    WITH med AS (
        SELECT parse_filename(filename, true) AS city,
        MEDIAN(price) AS median_price
        FROM read_csv_auto('../data/*.csv', filename = true, union_by_name = true)
        WHERE room_type='Entire home/apt'
        AND number_of_reviews_ltm >0
        GROUP BY parse_filename(filename,true)
        HAVING COUNT(*)>= 500)

    SELECT 
        c.city,
        c.price_sqm,
        m.median_price
    FROM med m
    JOIN cities c ON m.city=c.city
    ORDER BY c.city DESC
    LIMIT 50""").df()

,city,price_sqm,median_price
0,winnipeg,2341.75,160.0
1,washington-dc,8615.59,238.0
2,vienna,14565.33,125.0
3,victoria,5407.17,297.0
4,vancouver,8985.48,382.0
5,valencia,5677.61,166.0
6,twin-cities-msa,3395.59,264.0
7,toronto,8484.34,296.0
8,tokyo,10495.88,20700.0
9,thessaloniki,3859.51,74.0


In [53]:
duckdb.sql("""
SELECT
    regexp_extract(column0, '(\\d{4}-\\d{2}-\\d{2})', 1) AS snapshot_date,
    COUNT(*) AS n
FROM read_csv('../data/reference/city_urls.txt', header = false, columns = {'column0': 'VARCHAR'})
GROUP BY 1
ORDER BY 1
""").df()

,snapshot_date,n
0,2026-06-14,4
1,2026-06-15,15
2,2026-06-16,5
3,2026-06-19,5
4,2026-06-20,3
5,2026-06-21,5
6,2026-06-22,5
7,2026-06-23,5
8,2026-06-24,6
9,2026-06-25,3


In [54]:
import pandas as pd
r = pd.read_json("https://api.frankfurter.dev/v1/2026-06-30?base=USD")
r.head(20)

HTTPError: HTTP Error 403: Forbidden

In [55]:
import requests
r = requests.get("https://api.frankfurter.dev/v1/2026-06-30?base=USD")
print(r.status_code)
print(r.json())

200
{'amount': 1.0, 'base': 'USD', 'date': '2026-06-30', 'rates': {'AUD': 1.452, 'BRL': 5.1784, 'CAD': 1.4236, 'CHF': 0.80955, 'CNY': 6.7855, 'CZK': 21.288, 'DKK': 6.5599, 'EUR': 0.87765, 'GBP': 0.75635, 'HKD': 7.8418, 'HUF': 312.71, 'IDR': 17903, 'ILS': 2.9799, 'INR': 94.66, 'ISK': 126.38, 'JPY': 162.44, 'KRW': 1550.89, 'MXN': 17.468, 'MYR': 4.085, 'NOK': 9.9267, 'NZD': 1.7672, 'PHP': 61.358, 'PLN': 3.77, 'RON': 4.6023, 'SEK': 9.7363, 'SGD': 1.2949, 'THB': 33.23, 'TRY': 46.66, 'ZAR': 16.3721}}


In [56]:
duckdb.sql("SELECT city, numbeo_city FROM cities ORDER BY numbeo_city").df().to_csv("../data/reference/city_currency.csv", index=False)

In [57]:
duckdb.sql("""
    SELECT trim(split_part(numbeo_city, ',' , -1)) AS country,
           COUNT(*) AS n_cities
    FROM cities
    GROUP BY country
    ORDER BY n_cities DESC""").df()

,country,n_cities
0,United States,23
1,Canada,7
2,Italy,6
3,Spain,5
4,United Kingdom,4
5,Australia,3
6,France,3
7,Belgium,3
8,Netherlands,3
9,Greece,2


In [58]:
import requests, pandas as pd

api = requests.get("https://api.frankfurter.dev/v1/2026-06-30?base=USD",
                   headers={"User-Agent": "Mozilla/5.0"}).json()["rates"]

manual = {"COP": 3443.59, "ARS": 1450.00, "CLP": 922.34, "KES": 129.41, "TWD": 31.85}

rates = {**api, **manual, "USD": 1.0}

pd.DataFrame(rates.items(), columns=["currency", "usd_rate"]) \
  .to_csv("../data/reference/exchange_rates.csv", index=False)

In [59]:
duckdb.sql("SELECT * FROM read_csv_auto('../data/reference/exchange_rates.csv')").df()

,currency,usd_rate
0,AUD,1.45200
1,BRL,5.17840
2,CAD,1.42360
3,CHF,0.80955
4,CNY,6.78550
5,CZK,21.28800
6,DKK,6.55990
7,EUR,0.87765
8,GBP,0.75635
9,HKD,7.84180


In [60]:
duckdb.sql("""
CREATE OR REPLACE TABLE city_metrics AS
WITH med AS (
    SELECT
        parse_filename(filename, true) AS city,
        MEDIAN(price) AS med_price_local
    FROM read_csv_auto('../data/*.csv', filename = true, union_by_name = true)
    WHERE room_type = 'Entire home/apt'
      AND number_of_reviews_ltm > 0
    GROUP BY parse_filename(filename, true)
),
cur AS (
    SELECT * FROM read_csv_auto('../data/reference/country_currency.csv')
),
fx AS (
    SELECT * FROM read_csv_auto('../data/reference/exchange_rates.csv')
)
SELECT
    c.city,
    m.med_price_local,
    cur.currency,
    ROUND(m.med_price_local / fx.usd_rate, 2) AS med_price_usd,
    c.price_sqm
FROM cities c
JOIN med m           ON m.city = c.city
JOIN cur             ON cur.country = trim(split_part(c.numbeo_city, ',', -1))
JOIN fx              ON fx.currency = cur.currency
""")

In [61]:
duckdb.sql("SELECT * FROM city_metrics ORDER BY med_price_usd DESC").df()

,city,med_price_local,currency,med_price_usd,price_sqm
0,san-diego,389.0,USD,389.00,8093.34
1,amsterdam,331.0,EUR,377.14,10705.03
2,edinburgh,277.0,GBP,366.23,6689.03
3,boston,350.0,USD,350.00,13709.79
4,jersey-city,339.0,USD,339.00,7828.50
...,...,...,...,...,...
79,santiago,64547.0,CLP,69.98,2845.32
80,sao-paulo,341.0,BRL,65.85,2777.60
81,bangkok,1742.0,THB,52.42,6135.34
82,bogota,179900.0,COP,52.24,2665.91


In [62]:
duckdb.sql("SELECT * FROM city_metrics ORDER BY med_price_usd DESC LIMIT 50").df()

,city,med_price_local,currency,med_price_usd,price_sqm
0,san-diego,389.0,USD,389.00,8093.34
1,amsterdam,331.0,EUR,377.14,10705.03
2,edinburgh,277.0,GBP,366.23,6689.03
3,boston,350.0,USD,350.00,13709.79
4,jersey-city,339.0,USD,339.00,7828.50
5,seattle,314.0,USD,314.00,7592.28
6,london,236.0,GBP,312.02,20204.52
7,nashville,296.0,USD,296.00,7910.00
8,los-angeles,294.0,USD,294.00,7313.97
9,dublin,257.0,EUR,292.83,8988.30


In [63]:
duckdb.sql("SELECT * FROM city_metrics ORDER BY price_sqm DESC LIMIT 50").df()

,city,med_price_local,currency,med_price_usd,price_sqm
0,hong-kong,1069.0,HKD,136.32,28216.99
1,london,236.0,GBP,312.02,20204.52
2,new-york-city,208.0,USD,208.00,18759.46
3,paris,212.0,EUR,241.55,15008.74
4,vienna,125.0,EUR,142.43,14565.33
5,boston,350.0,USD,350.00,13709.79
6,munich,188.5,EUR,214.78,13162.17
7,sydney,343.0,AUD,236.23,12960.02
8,taipei,2977.5,TWD,93.49,12706.98
9,copenhagen,1752.0,DKK,267.08,11806.49


In [67]:
duckdb.sql("""
    SELECT city,
           med_price_usd,
           price_sqm,
           ROUND(100.0*med_price_usd*182 / (price_sqm*60), 2) AS roi_pct
    FROM city_metrics
    ORDER BY roi_pct DESC""").df()

,city,med_price_usd,price_sqm,roi_pct
0,rochester,197.00,1462.56,40.86
1,twin-cities-msa,264.00,3395.59,23.58
2,chicago,265.00,4094.16,19.63
3,columbus,204.00,3440.00,17.99
4,fort-worth,239.50,4109.17,17.68
...,...,...,...,...
79,new-york-city,208.00,18759.46,3.36
80,vienna,142.43,14565.33,2.97
81,bangkok,52.42,6135.34,2.59
82,taipei,93.49,12706.98,2.23


In [69]:
duckdb.sql(""" DESCRIBE SELECT * FROM read_csv_auto('../data/reference/numbeo_utilities.csv')""").df()

,column_name,column_type,null,key,default,extra
0,rank,BIGINT,YES,None,None,None
1,city,VARCHAR,YES,None,None,None
2,price_utilities,DOUBLE,YES,None,None,None


In [80]:
duckdb.sql("""
CREATE OR REPLACE TABLE city_metrics AS 
    WITH med AS (SELECT parse_filename(filename,true) AS city,
                    MEDIAN (price) AS median_price_local
                FROM read_csv_auto('../data/*.csv', filename = true, union_by_name = true)
                WHERE room_type = 'Entire home/apt'
                AND number_of_reviews_ltm >0
                GROUP BY parse_filename(filename, true)),

    cur AS (SELECT * FROM read_csv_auto('../data/reference/country_currency.csv', filename = true, union_by_name = true)),
    ex AS (SELECT * FROM read_csv_auto('../data/reference/exchange_rates.csv', filename = true, union_by_name = true)),
    out AS (SELECT city AS numbeo_city, price_sqm AS price_sqm_out FROM read_csv_auto('../data/reference/numbeo_outside.csv')),
    utl AS (SELECT * FROM read_csv_auto('../data/reference/numbeo_utilities.csv', filename = true, union_by_name = true))

    SELECT
        c.city,
        c.numbeo_city,
        ROUND(m.median_price_local / ex.usd_rate, 2) AS med_airbnb_price_usd,
        c.price_sqm AS price_sqm_centre,
        out.price_sqm_out,
        utl.price_utilities
    FROM cities c
    JOIN med m ON m.city=c.city
    JOIN cur ON cur.country=trim(split_part(c.numbeo_city, ',' , -1))
    JOIN ex ON ex.currency=cur.currency
    LEFT JOIN out ON out.numbeo_city=c.numbeo_city
    LEFT JOIN utl ON utl.city=c.numbeo_city
    """)


In [72]:
duckdb.sql("""SHOW TABLES""").df()

,name
0,cities
1,city_metrics


In [73]:
duckdb.sql("""DESCRIBE cities""").df()

,column_name,column_type,null,key,default,extra
0,city,VARCHAR,YES,None,None,None
1,active_entire_homes,BIGINT,YES,None,None,None
2,numbeo_city,VARCHAR,YES,None,None,None
3,price_sqm,DOUBLE,YES,None,None,None


In [81]:
duckdb.sql("""SELECT * FROM city_metrics""").df()

,city,numbeo_city,med_airbnb_price_usd,price_sqm_centre,price_sqm_out,price_utilities
0,hong-kong,"Hong Kong, Hong Kong (China)",136.32,28216.99,17416.19,230.14
1,paris,"Paris, France",241.55,15008.74,10137.60,266.43
2,santa-clara-county,"San Jose, CA, United States",289.00,8725.79,9048.77,274.03
3,munich,"Munich, Germany",214.78,13162.17,9043.45,381.66
4,san-francisco,"San Francisco, CA, United States",288.00,10009.83,8635.32,236.26
...,...,...,...,...,...,...
79,cape-town,"Cape Town, South Africa",108.05,2388.51,1594.66,125.58
80,rio-de-janeiro,"Rio de Janeiro, Brazil",85.93,3674.84,1508.01,122.63
81,rochester,"Rochester, NY, United States",197.00,1462.56,1462.56,144.44
82,new-orleans,"New Orleans, LA, United States",215.50,5200.75,1379.00,261.99


In [79]:
duckdb.sql(""" 
    SELECT 
        COUNT (*) AS n,
        SUM(CASE WHEN price_sqm_out IS NULL THEN 1 ELSE 0 END) AS missing_price_sqm_out,
        SUM(CASE WHEN price_utilities IS NULL THEN 1 ELSE 0 END) AS missing_price_utilities
    FROM city_metrics
    """    )

┌───────┬───────────────────────┬─────────────────────────┐
│   n   │ missing_price_sqm_out │ missing_price_utilities │
│ int64 │        int128         │         int128          │
├───────┼───────────────────────┼─────────────────────────┤
│    84 │                     0 │                       0 │
└───────┴───────────────────────┴─────────────────────────┘

In [85]:
duckdb.sql("""SELECT city,
    price_sqm_centre,
    price_sqm_out,
    ROUND(price_sqm_out/price_sqm_centre,2) AS price_sqm_ratio
FROM city_metrics
ORDER BY price_sqm_ratio DESC
""").df()

,city,price_sqm_centre,price_sqm_out,price_sqm_ratio
0,ottawa,4368.09,4575.34,1.05
1,santa-clara-county,8725.79,9048.77,1.04
2,rochester,1462.56,1462.56,1.00
3,winnipeg,2341.75,2345.49,1.00
4,santiago,2845.32,2738.78,0.96
...,...,...,...,...
79,washington-dc,8615.59,3254.67,0.38
80,new-york-city,18759.46,6523.13,0.35
81,nashville,7910.00,2687.00,0.34
82,dallas,9410.00,3181.49,0.34


In [87]:
duckdb.sql("""
SELECT
    parse_filename(filename, true) AS city,
    MEDIAN(minimum_nights) AS med_min_nights
FROM read_csv_auto('../data/*.csv', filename = true, union_by_name = true)
WHERE room_type = 'Entire home/apt'
  AND number_of_reviews_ltm > 0
GROUP BY parse_filename(filename, true)
ORDER BY med_min_nights DESC
LIMIT 50
""").df()

,city,med_min_nights
0,new-york-city,30.0
1,singapore,6.0
2,san-francisco,3.0
3,ottawa,3.0
4,vienna,2.0
5,toronto,2.0
6,barwon-south-west-vic,2.0
7,portland,2.0
8,south-aegean,2.0
9,clark-county-nv,2.0


In [94]:
duckdb.sql("""
WITH listing_nights AS (
    SELECT
        parse_filename(filename, true) AS city,
        LEAST((number_of_reviews_ltm / 0.5) * GREATEST(3, minimum_nights), 255) AS nights_booked,
        (number_of_reviews_ltm / 0.5) * GREATEST(3, minimum_nights) AS nights_uncapped
    FROM read_csv_auto('../data/*.csv', filename = true, union_by_name = true)
    WHERE room_type = 'Entire home/apt'
      AND number_of_reviews_ltm > 0
)
SELECT
    ln.city,
    ROUND(MEDIAN(ln.nights_booked), 0) AS med_nights,
    ROUND(100.0 * SUM(CASE WHEN ln.nights_uncapped > 255 THEN 1 ELSE 0 END) / COUNT(*), 1) AS pct_capped
FROM listing_nights ln
JOIN cities c ON c.city = ln.city
GROUP BY ln.city
ORDER BY med_nights DESC
LIMIT 50
""").df()

,city,med_nights,pct_capped
0,ottawa,156.0,26.7
1,quebec-city,150.0,28.1
2,victoria,144.0,22.7
3,portland,144.0,22.1
4,barcelona,138.0,20.3
5,chicago,128.0,16.4
6,edinburgh,126.0,25.2
7,budapest,126.0,24.0
8,boston,126.0,17.8
9,winnipeg,126.0,19.6


In [95]:
duckdb.sql("""SHOW TABLES""").df()

,name
0,cities
1,city_metrics


In [96]:
duckdb.sql("""SHOW TABLE city_metrics""").df()

,column_name,column_type,null,key,default,extra
0,city,VARCHAR,YES,None,None,None
1,numbeo_city,VARCHAR,YES,None,None,None
2,med_airbnb_price_usd,DOUBLE,YES,None,None,None
3,price_sqm_centre,DOUBLE,YES,None,None,None
4,price_sqm_out,DOUBLE,YES,None,None,None
5,price_utilities,DOUBLE,YES,None,None,None


In [102]:
duckdb.sql("SELECT city,city_display, med_airbnb_price_usd, med_nights, pct_capped, roi_centre, roi_outside FROM city_metrics ORDER BY roi_centre DESC").df()

,city,city_display,med_airbnb_price_usd,med_nights,pct_capped,roi_centre,roi_outside
0,rochester,Rochester,197.00,96.0,10.3,19.16,19.16
1,chicago,Chicago,265.00,128.0,16.4,12.14,18.95
2,twin-cities-msa,Minneapolis,264.00,90.0,9.8,10.00,16.18
3,columbus,Columbus,204.00,120.0,20.9,9.95,20.21
4,edinburgh,Edinburgh,366.23,126.0,25.2,9.71,14.31
...,...,...,...,...,...,...,...
78,london,London,312.02,42.0,4.8,-0.18,0.94
79,nairobi,Nairobi,47.96,24.0,2.1,-0.22,0.25
80,munich,Munich,214.78,36.0,7.6,-0.43,-0.17
81,hong-kong,Hong Kong,136.32,54.0,6.4,-0.68,-0.48


In [103]:
duckdb.sql("SELECT city,city_display, med_airbnb_price_usd, med_nights, pct_capped, roi_centre, roi_outside FROM city_metrics ORDER BY roi_centre DESC LIMIT 50").df()

,city,city_display,med_airbnb_price_usd,med_nights,pct_capped,roi_centre,roi_outside
0,rochester,Rochester,197.00,96.0,10.3,19.16,19.16
1,chicago,Chicago,265.00,128.0,16.4,12.14,18.95
2,twin-cities-msa,Minneapolis,264.00,90.0,9.8,10.00,16.18
3,columbus,Columbus,204.00,120.0,20.9,9.95,20.21
4,edinburgh,Edinburgh,366.23,126.0,25.2,9.71,14.31
5,fort-worth,Fort Worth,239.50,114.0,15.4,9.35,22.30
6,portland,Portland,180.00,144.0,22.1,9.03,10.59
7,san-diego,San Diego,389.00,120.0,14.4,8.19,10.31
8,winnipeg,Winnipeg,112.39,126.0,19.6,8.06,8.04
9,victoria,Victoria,208.63,144.0,22.7,7.98,9.82


In [104]:
duckdb.sql("""
SELECT neighbourhood, COUNT(*) AS n
FROM read_csv_auto('../data/clark-county-nv.csv')
GROUP BY 1 ORDER BY 2 DESC LIMIT 10
""").df()

,neighbourhood,n
0,Unincorporated Areas,15821
1,City of Las Vegas,2498
2,City of Henderson,897
3,City of North Las Vegas,868
4,City of Mesquite,187
5,Boulder City,19
6,Nellis AFB,6


In [105]:
urls = open('../data/reference/city_urls.txt').read().splitlines()
keep = duckdb.sql("SELECT city FROM cities").df()["city"].tolist()

out = []
for u in urls:
    name = u.split("/")[-4]
    if name in keep:
        out.append(u.replace("visualisations/listings.csv", "data/listings.csv.gz"))

open('../data/reference/detailed_urls.txt', 'w').write("\n".join(out))
print(len(out))

UnicodeDecodeError: 'charmap' codec can't decode byte 0x8d in position 5273: character maps to <undefined>

In [106]:
urls = open('../data/reference/city_urls.txt', encoding="utf-8").read().splitlines()
keep = duckdb.sql("SELECT city FROM cities").df()["city"].tolist()

out = []
for u in urls:
    name = u.split("/")[-4]
    if name in keep:
        out.append(u.replace("visualisations/listings.csv", "data/listings.csv.gz"))

open('../data/reference/detailed_urls.txt', 'w', encoding="utf-8").write("\n".join(out))
print(len(out))

82


In [107]:
print(out[0])

https://data.insideairbnb.com/argentina/ciudad-autónoma-de-buenos-aires/buenos-aires/2026-06-29/data/listings.csv.gz


In [108]:
matched = [u.split("/")[-4] for u in urls if u.split("/")[-4] in keep]
missing = [c for c in keep if c not in matched]
print(missing)

['bogota', 'sao-paulo']


In [109]:
import unicodedata

def clean(s):
    return unicodedata.normalize("NFKD", s).encode("ascii", "ignore").decode()

urls = open('../data/reference/city_urls.txt', encoding="utf-8").read().splitlines()
keep = duckdb.sql("SELECT city FROM cities").df()["city"].tolist()

out = []
for u in urls:
    name = clean(u.split("/")[-4])
    if name in keep:
        out.append(u.replace("visualisations/listings.csv", "data/listings.csv.gz"))

open('../data/reference/detailed_urls.txt', 'w', encoding="utf-8").write("\n".join(out))
print(len(out))

84


In [110]:
names = [clean(u.split("/")[-4]) for u in urls]
print([n for n in names if names.count(n) > 1])

[]


In [111]:
print(out[0])

https://data.insideairbnb.com/argentina/ciudad-autónoma-de-buenos-aires/buenos-aires/2026-06-29/data/listings.csv.gz


In [112]:
import requests, unicodedata, pathlib
from urllib.parse import quote

pathlib.Path('../data/detailed').mkdir(exist_ok=True)

for i, u in enumerate(out, 1):
    name = clean(u.split("/")[-4])
    dest = pathlib.Path(f'../data/detailed/{name}.csv.gz')
    if dest.exists():
        continue
    r = requests.get(quote(u, safe=":/"), headers={"User-Agent": "Mozilla/5.0"})
    if r.status_code == 200:
        dest.write_bytes(r.content)
        print(i, name, len(r.content) // 1024, "KB")
    else:
        print(i, name, "FAILED", r.status_code)

1 buenos-aires 16214 KB
2 sydney 12337 KB
3 brisbane 3750 KB
4 melbourne 14912 KB
5 vienna 7171 KB
6 brussels 2750 KB
7 antwerp 1229 KB
8 ghent 728 KB
9 rio-de-janeiro 23183 KB
10 sao-paulo 23855 KB
11 vancouver 3948 KB
12 victoria 2217 KB
13 winnipeg 991 KB
14 ottawa 1529 KB
15 toronto 12428 KB
16 montreal 5693 KB
17 quebec-city 1193 KB
18 santiago 10394 KB
19 hong-kong 2901 KB
20 bogota 9915 KB
21 prague 6240 KB
22 copenhagen 11422 KB
23 lyon 4639 KB
24 paris 40122 KB
25 bordeaux 6399 KB
26 berlin 6620 KB
27 munich 3328 KB
28 athens 8756 KB
29 thessaloniki 2684 KB
30 budapest 6319 KB
31 dublin 3468 KB
32 naples 5781 KB
33 bologna 2626 KB
34 rome 21304 KB
35 bergamo 2413 KB
36 milan 15033 KB
37 florence 7684 KB
38 tokyo 26359 KB
39 nairobi 9408 KB
40 riga 1808 KB
41 mexico-city 17980 KB
42 oslo 6933 KB
43 lisbon 13938 KB
44 porto 9238 KB
45 cape-town 15744 KB
46 malaga 5312 KB
47 sevilla 5063 KB
48 barcelona 7984 KB
49 madrid 10125 KB
50 valencia 4261 KB
51 stockholm 2498 KB
52 taipei

In [113]:
import pathlib
got = [p.stem.replace('.csv','') for p in pathlib.Path('../data/detailed').glob('*.csv.gz')]
print(len(got), "downloaded")
print("missing:", [c for c in keep if c not in got])

84 downloaded
missing: []


In [114]:
duckdb.sql(""" SELECT * FROM read_csv_auto('../data/detailed/paris.csv.gz')""").df()

,id,listing_url,scrape_id,last_scraped,source,name,description,neighborhood_overview,picture_url,host_id,...,review_scores_communication,review_scores_location,review_scores_value,license,instant_bookable,calculated_host_listings_count,calculated_host_listings_count_entire_homes,calculated_host_listings_count_private_rooms,calculated_host_listings_count_shared_rooms,reviews_per_month
0,114543,https://www.airbnb.com/rooms/114543,20260616211535,2026-06-21,city scrape,Charming 55m² flat - Eiffel Tower,Fully furnished charming flat with lots of spa...,None,https://a0.muscache.com/pictures/airflow/Hosti...,581233,...,4.97,4.93,4.77,7511502831082,None,1,1,0,0,0.83
1,115107,https://www.airbnb.com/rooms/115107,20260616211535,2026-06-25,previous scrape,Charming two-bed apartment with balcony,This charming 60 square meter apartment is per...,None,https://a0.muscache.com/pictures/miso/Hosting-...,379042,...,5.00,4.40,4.40,7511002008206,None,2,1,1,0,0.22
2,115655,https://www.airbnb.com/rooms/115655,20260616211535,2026-06-20,city scrape,Family loft with a private courtyard,"Beautiful openplan loft situated in a private,...",None,https://a0.muscache.com/pictures/33160696/6171...,584977,...,4.69,4.04,4.40,7511805457226,None,1,1,0,0,1.24
3,120494,https://www.airbnb.com/rooms/120494,20260616211535,2026-06-19,city scrape,Romantic Hideout in Paris,"""As full of spirit as the month of May, as gor...",None,https://a0.muscache.com/pictures/833324/f66a6b...,606427,...,4.97,4.85,4.75,7511300905786,None,1,1,0,0,2.03
4,125686,https://www.airbnb.com/rooms/125686,20260616211535,2026-06-20,city scrape,Studio - République/Canal Saint Martin,"Tiny studio in central Paris, between the Mara...",None,https://a0.muscache.com/pictures/2704337/e28df...,624523,...,4.92,4.87,4.69,"Available with a mobility lease only (""bail mo...",None,1,1,0,0,0.79
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
77674,1704247796863989884,https://www.airbnb.com/rooms/1704247796863989884,20260616211535,2026-06-27,city scrape,Studio central Opéra Garnier - Paris II,This charming studio is comfortable and conven...,None,https://a0.muscache.com/pictures/hosting/Hosti...,519533271,...,NaN,NaN,NaN,Exempt - hotel-type listing,None,52,52,0,0,NaN
77675,1704247801184632532,https://www.airbnb.com/rooms/1704247801184632532,20260616211535,2026-06-27,city scrape,Magnifique loft 130m2 Louvre/Palais Royal - IV,"Discover this magnificent loft of 130m2, compo...",None,https://a0.muscache.com/pictures/hosting/Hosti...,519533271,...,NaN,NaN,NaN,Exempt - hotel-type listing,None,52,52,0,0,NaN
77676,1704247804216920178,https://www.airbnb.com/rooms/1704247804216920178,20260616211535,2026-06-20,city scrape,Appartement agréable Montmartre - Sacré-Cœur II,This charming apartment is located in the hear...,None,https://a0.muscache.com/pictures/hosting/Hosti...,519533271,...,NaN,NaN,NaN,7511809055442,None,52,52,0,0,NaN
77677,1704248328813864955,https://www.airbnb.com/rooms/1704248328813864955,20260616211535,2026-06-21,city scrape,Petit studio pratique proche Sacré-Cœur II,This very small and charming studio is located...,None,https://a0.muscache.com/pictures/hosting/Hosti...,262969486,...,NaN,NaN,NaN,7510413688666,None,34,34,0,0,NaN


In [115]:
duckdb.sql(""" 
SELECT id, price, bedrooms, accommodates, property_type, room_type, minimum_nights, number_of_reviews_ltm
FROM read_csv_auto('../data/detailed/paris.csv.gz')""").df()

,id,price,bedrooms,accommodates,property_type,room_type,minimum_nights,number_of_reviews_ltm
0,114543,$226.75,1,2,Entire rental unit,Entire home/apt,4,6
1,115107,None,2,4,Entire rental unit,Entire home/apt,3,0
2,115655,$225.00,2,4,Entire loft,Entire home/apt,1,13
3,120494,$95.33,1,2,Entire rental unit,Entire home/apt,3,8
4,125686,$37.40,1,1,Entire rental unit,Entire home/apt,30,1
...,...,...,...,...,...,...,...,...
77674,1704247796863989884,$186.00,<NA>,2,Entire rental unit,Entire home/apt,1,0
77675,1704247801184632532,$406.00,2,4,Entire rental unit,Entire home/apt,1,0
77676,1704247804216920178,$142.00,1,4,Entire rental unit,Entire home/apt,1,0
77677,1704248328813864955,$87.00,<NA>,2,Entire rental unit,Entire home/apt,1,0


In [120]:
duckdb.sql("""SELECT COUNT(*) AS n,bedrooms, property_type, room_type, price
FROM read_csv_auto('../data/detailed/paris.csv.gz')
WHERE bedrooms IS NOT NULL 
GROUP BY bedrooms""").df()

BinderException: Binder Error: column "property_type" must appear in the GROUP BY clause or must be part of an aggregate function.
Either add it to the GROUP BY list, or use "ANY_VALUE(property_type)" if the exact value of "property_type" is not important.

In [122]:
duckdb.sql("""SELECT bedrooms, COUNT(*) AS n,
FROM read_csv_auto('../data/detailed/paris.csv.gz')
WHERE room_type = 'Entire home/apt'
GROUP BY bedrooms
ORDER BY bedrooms""").df()

,bedrooms,n
0,1,40046
1,2,13749
2,3,4873
3,4,1262
4,5,251
5,6,61
6,7,19
7,8,4
8,10,3
9,11,1


In [123]:
duckdb.sql("""
SELECT bedrooms, MEDIAN(accommodates) AS med_acc, COUNT(*) AS n
FROM read_csv_auto('../data/detailed/paris.csv.gz')
WHERE room_type = 'Entire home/apt' AND bedrooms IS NOT NULL
GROUP BY bedrooms ORDER BY bedrooms
""").df()

,bedrooms,med_acc,n
0,1,2.0,40046
1,2,4.0,13749
2,3,6.0,4873
3,4,8.0,1262
4,5,10.0,251
5,6,12.0,61
6,7,14.0,19
7,8,8.0,4
8,10,16.0,3
9,11,16.0,1


In [125]:
duckdb.sql(""" 
SELECT parse_filename(filename, true) AS city,
COUNT(*) AS n,
ROUND(100.0*SUM(CASE WHEN bedrooms IS NULL THEN 1 ELSE 0 END)/ COUNT(*),1 ) AS pct_missing_bedrooms
FROM read_csv_auto('../data/detailed/*.csv.gz', filename = true, union_by_name = true)
WHERE room_type = 'Entire home/apt'
AND number_of_reviews_ltm >0
GROUP BY 1
ORDER BY pct_missing_bedrooms DESC
LIMIT 50
""").df()

,city,n,pct_missing_bedrooms
0,new-york-city.csv,5763,18.5
1,boston.csv,1943,16.7
2,buenos-aires.csv,20895,14.8
3,hong-kong.csv,1297,14.6
4,riga.csv,2520,14.4
5,lyon.csv,4100,13.9
6,porto.csv,10360,12.7
7,oakland.csv,1166,12.6
8,paris.csv,37404,12.5
9,san-francisco.csv,2843,12.1


In [131]:
duckdb.sql("""
CREATE OR REPLACE TABLE city_metrics AS
WITH listing_nights AS (
    SELECT
        parse_filename(filename, true) AS city,
        CAST(replace(replace(price, '$', ''), ',', '') AS DOUBLE) AS price_num,
        LEAST((number_of_reviews_ltm / 0.5) * GREATEST(3, minimum_nights), 255) AS nights_booked,
        (number_of_reviews_ltm / 0.5) * GREATEST(3, minimum_nights) AS nights_uncapped
    FROM read_csv_auto('../data/detailed/*.csv.gz', filename = true, union_by_name = true)
    WHERE room_type = 'Entire home/apt'
      AND number_of_reviews_ltm > 0
      AND bedrooms <= 2
),
agg AS (
    SELECT
        replace(city, '.csv', '') AS city,
        MEDIAN(price_num) AS med_price_local,
        MEDIAN(nights_booked) AS med_nights,
        ROUND(100.0 * SUM(CASE WHEN nights_uncapped > 255 THEN 1 ELSE 0 END) / COUNT(*), 1) AS pct_capped,
        COUNT(*) AS n_listings
    FROM listing_nights
    GROUP BY 1
    HAVING COUNT(*) >= 500
),
cur AS (SELECT * FROM read_csv_auto('../data/reference/country_currency.csv')),
ex  AS (SELECT * FROM read_csv_auto('../data/reference/exchange_rates.csv')),
out AS (SELECT city AS numbeo_city, price_sqm AS price_sqm_out
        FROM read_csv_auto('../data/reference/numbeo_outside.csv')),
utl AS (SELECT city AS numbeo_city, price_utilities
        FROM read_csv_auto('../data/reference/numbeo_utilities.csv'))
SELECT
    c.city,
    split_part(c.numbeo_city, ',', 1) AS city_display,
    a.n_listings,
    ROUND(a.med_price_local / ex.usd_rate, 2) AS med_airbnb_price_usd,
    ROUND(a.med_nights, 0) AS med_nights,
    a.pct_capped,
    c.price_sqm AS price_sqm_centre,
    out.price_sqm_out,
    utl.price_utilities,
    ROUND(100.0 * (a.med_price_local / ex.usd_rate * a.med_nights) / (c.price_sqm * 60)
          - 100.0 * (12 * utl.price_utilities) / (85 * c.price_sqm) - 1.0, 2) AS roi_centre,
    ROUND(100.0 * (a.med_price_local / ex.usd_rate * a.med_nights) / (out.price_sqm_out * 60)
          - 100.0 * (12 * utl.price_utilities) / (85 * out.price_sqm_out) - 1.0, 2) AS roi_outside
FROM cities c
JOIN agg a   ON a.city = c.city
JOIN cur     ON cur.country = trim(split_part(c.numbeo_city, ',', -1))
JOIN ex      ON ex.currency = cur.currency
LEFT JOIN out ON out.numbeo_city = c.numbeo_city
LEFT JOIN utl ON utl.numbeo_city = c.numbeo_city
WHERE c.city <> 'new-york-city'
""")

In [133]:
duckdb.sql("SELECT city_display, n_listings, med_airbnb_price_usd, med_nights, roi_centre, roi_outside FROM city_metrics ORDER BY roi_centre DESC LIMIT 20 ").df()

,city_display,n_listings,med_airbnb_price_usd,med_nights,roi_centre,roi_outside
0,Chicago,2935,225.18,138.0,10.98,17.19
1,Edinburgh,2900,331.86,138.0,9.63,14.19
2,Portland,2009,177.23,144.0,8.86,10.40
3,Columbus,1112,167.00,132.0,8.77,17.92
4,Minneapolis,1763,197.37,102.0,8.22,13.40
5,Victoria,1694,187.58,156.0,7.74,9.53
6,Fort Worth,734,192.50,120.0,7.65,18.47
7,Barcelona,4953,245.45,150.0,6.71,10.22
8,Winnipeg,813,96.23,126.0,6.61,6.60
9,San Diego,4806,303.38,126.0,6.45,8.17


In [130]:
duckdb.sql("""
SELECT
    'detailed, all bedrooms' AS variant,
    MEDIAN(LEAST((number_of_reviews_ltm / 0.5) * GREATEST(3, minimum_nights), 255)) AS med_nights,
    MEDIAN(minimum_nights) AS med_min_nights,
    MEDIAN(number_of_reviews_ltm) AS med_reviews,
    COUNT(*) AS n
FROM read_csv_auto('../data/detailed/london.csv.gz')
WHERE room_type = 'Entire home/apt' AND number_of_reviews_ltm > 0
UNION ALL
SELECT
    'detailed, 1-2 bed',
    MEDIAN(LEAST((number_of_reviews_ltm / 0.5) * GREATEST(3, minimum_nights), 255)),
    MEDIAN(minimum_nights),
    MEDIAN(number_of_reviews_ltm),
    COUNT(*)
FROM read_csv_auto('../data/detailed/london.csv.gz')
WHERE room_type = 'Entire home/apt' AND number_of_reviews_ltm > 0 AND bedrooms <= 2
""").df()

,variant,med_nights,med_min_nights,med_reviews,n
0,"detailed, all bedrooms",42.0,2.0,6.0,33455
1,"detailed, 1-2 bed",42.0,2.0,6.0,25785


In [135]:
duckdb.sql("""SELECT
    split_part(c.numbeo_city, ',', 1) AS city,
    trim(split_part(c.numbeo_city, ',', -1)) AS country,
    c.city AS city_slug,
    a.n_listings,
    ROUND(a.med_price_local / ex.usd_rate, 2) AS med_airbnb_price_usd,
    ROUND(a.med_nights, 0) AS med_nights,
    ROUND(100.0 * a.med_nights / 365, 1) AS occupancy_pct,
    ROUND(a.med_price_local / ex.usd_rate * a.med_nights, 0) AS annual_revenue_usd,
    a.pct_capped,
    c.price_sqm AS price_sqm_centre,
    out.price_sqm_out,
    utl.price_utilities,
    ROUND(100.0 * (a.med_price_local / ex.usd_rate * a.med_nights) / (c.price_sqm * 60)
          - 100.0 * (12 * utl.price_utilities) / (85 * c.price_sqm) - 1.0, 2) AS roi_centre,
    ROUND(100.0 * (a.med_price_local / ex.usd_rate * a.med_nights) / (out.price_sqm_out * 60)
          - 100.0 * (12 * utl.price_utilities) / (85 * out.price_sqm_out) - 1.0, 2) AS roi_outside""").df()

BinderException: Binder Error: Referenced table "c" not found!

In [7]:
duckdb.sql("""
CREATE OR REPLACE TABLE city_metrics AS
WITH listing_nights AS (
    SELECT
        parse_filename(filename, true) AS city,
        CAST(replace(replace(price, '$', ''), ',', '') AS DOUBLE) AS price_num,
        LEAST((number_of_reviews_ltm / 0.5) * GREATEST(3, minimum_nights), 255) AS nights_booked,
        (number_of_reviews_ltm / 0.5) * GREATEST(3, minimum_nights) AS nights_uncapped
    FROM read_csv_auto('../data/detailed/*.csv.gz', filename = true, union_by_name = true)
    WHERE room_type = 'Entire home/apt'
      AND number_of_reviews_ltm > 0
      AND bedrooms <= 2
),
agg AS (
    SELECT
        replace(city, '.csv', '') AS city,
        MEDIAN(price_num) AS med_price_local,
        MEDIAN(nights_booked) AS med_nights,
        ROUND(100.0 * SUM(CASE WHEN nights_uncapped > 255 THEN 1 ELSE 0 END) / COUNT(*), 1) AS pct_capped,
        COUNT(*) AS n_listings
    FROM listing_nights
    GROUP BY 1
    HAVING COUNT(*) >= 500
),
cur AS (SELECT * FROM read_csv_auto('../data/reference/country_currency.csv')),
ex  AS (SELECT * FROM read_csv_auto('../data/reference/exchange_rates.csv')),
out AS (SELECT city AS numbeo_city, price_sqm AS price_sqm_out
        FROM read_csv_auto('../data/reference/numbeo_outside.csv')),
utl AS (SELECT city AS numbeo_city, price_utilities
        FROM read_csv_auto('../data/reference/numbeo_utilities.csv'))
SELECT
    split_part(c.numbeo_city, ',', 1) AS city,
    trim(split_part(c.numbeo_city, ',', -1)) AS country,
    c.city AS city_slug,
    a.n_listings,
    ROUND(a.med_price_local / ex.usd_rate, 2) AS med_airbnb_price_usd,
    ROUND(a.med_nights, 0) AS med_nights,
    ROUND(100.0 * a.med_nights / 365, 1) AS occupancy_pct,
    ROUND(a.med_price_local / ex.usd_rate * a.med_nights, 0) AS annual_revenue_usd,
    a.pct_capped,
    c.price_sqm AS price_sqm_centre,
    out.price_sqm_out,
    utl.price_utilities,
    ROUND(100.0 * (a.med_price_local / ex.usd_rate * a.med_nights) / (c.price_sqm * 60)
          - 100.0 * (12 * utl.price_utilities) / (85 * c.price_sqm) - 1.0, 2) AS roi_centre,
    ROUND(100.0 * (a.med_price_local / ex.usd_rate * a.med_nights) / (out.price_sqm_out * 60)
          - 100.0 * (12 * utl.price_utilities) / (85 * out.price_sqm_out) - 1.0, 2) AS roi_outside
FROM cities c
JOIN agg a   ON a.city = c.city
JOIN cur     ON cur.country = trim(split_part(c.numbeo_city, ',', -1))
JOIN ex      ON ex.currency = cur.currency
LEFT JOIN out ON out.numbeo_city = c.numbeo_city
LEFT JOIN utl ON utl.numbeo_city = c.numbeo_city
WHERE c.city <> 'new-york-city'
""")

CatalogException: Catalog Error: Table with name cities does not exist!
Did you mean "pg_views"?

LINE 48: FROM cities c
              ^

In [140]:
duckdb.sql("""
SELECT city, country, occupancy_pct, annual_revenue_usd, roi_centre, roi_outside
FROM city_metrics ORDER BY annual_revenue_usd DESC LIMIT 20
""").df()

,city,country,occupancy_pct,annual_revenue_usd,roi_centre,roi_outside
0,Edinburgh,United Kingdom,37.8,45796.0,9.63,14.19
1,Boston,United States,34.5,40740.0,3.69,8.94
2,San Diego,United States,34.5,38225.0,6.45,8.17
3,Barcelona,Spain,41.1,36817.0,6.71,10.22
4,Seattle,United States,32.9,33720.0,5.86,9.33
5,Vancouver,Canada,36.2,32128.0,4.83,6.44
6,San Francisco,United States,32.9,31819.0,3.96,4.76
7,Jersey City,United States,31.2,31350.0,5.30,7.95
8,Chicago,United States,37.8,31075.0,10.98,17.19
9,Los Angeles,United States,32.9,29520.0,5.19,5.49


In [144]:
duckdb.sql("SELECT country, COUNT (*) AS n  FROM city_metrics GROUP BY country ORDER BY n DESC").df()

,country,n
0,United States,21
1,Canada,7
2,Italy,6
3,Spain,5
4,United Kingdom,4
5,Netherlands,3
6,Belgium,3
7,France,3
8,Australia,3
9,Brazil,2


In [146]:
duckdb.sql("""
SELECT city, country, roi_centre AS roi_at_60,
    ROUND(100.0 * (med_airbnb_price_usd * med_nights) / (price_sqm_centre * 85)
          - 100.0 * (12 * price_utilities) / (85 * price_sqm_centre) - 1.0, 2) AS roi_at_85
FROM city_metrics
ORDER BY roi_centre DESC 
LIMIT 50
""").df()

,city,country,roi_at_60,roi_at_85
0,Chicago,United States,10.98,7.26
1,Edinburgh,United Kingdom,9.63,6.27
2,Portland,United States,8.86,5.65
3,Columbus,United States,8.77,5.62
4,Minneapolis,United States,8.22,5.32
5,Victoria,Canada,7.74,5.08
6,Fort Worth,United States,7.65,4.89
7,Barcelona,Spain,6.71,4.34
8,Winnipeg,Canada,6.61,4.07
9,San Diego,United States,6.45,4.14


In [147]:
duckdb.sql("""
WITH r AS (
  SELECT city,
    RANK() OVER (ORDER BY roi_centre DESC) AS rank_60,
    RANK() OVER (ORDER BY
      100.0*(med_airbnb_price_usd*med_nights)/(price_sqm_centre*85)
      - 100.0*(12*price_utilities)/(85*price_sqm_centre) - 1.0 DESC) AS rank_85
  FROM city_metrics
)
SELECT MAX(ABS(rank_60 - rank_85)) AS biggest_move,
       ROUND(AVG(ABS(rank_60 - rank_85)), 2) AS avg_move
FROM r
""").df()

,biggest_move,avg_move
0,11,1.21


In [150]:
duckdb.sql("""
WITH r AS (
  SELECT city, country, roi_centre,
    RANK() OVER (ORDER BY roi_centre DESC) AS rank_60,
    RANK() OVER (ORDER BY
      100.0*(med_airbnb_price_usd*med_nights)/(price_sqm_centre*85)
      - 100.0*(12*price_utilities)/(85*price_sqm_centre) - 1.0 DESC) AS rank_85
  FROM city_metrics
)
SELECT city, country, roi_centre, rank_60, rank_85, rank_85 - rank_60 AS move
FROM r ORDER BY ABS(rank_85 - rank_60) DESC LIMIT 50
""").df()

,city,country,roi_centre,rank_60,rank_85,move
0,Riga,Latvia,0.29,67,78,11
1,Manchester,United Kingdom,2.23,39,44,5
2,Taipei,Taiwan,0.07,72,68,-4
3,Thessaloniki,Greece,0.12,70,74,4
4,Sydney,Australia,0.91,62,58,-4
5,Madrid,Spain,2.19,42,38,-4
6,Stockholm,Sweden,0.06,73,70,-3
7,Buenos Aires,Argentina,1.13,58,61,3
8,Melbourne,Australia,1.48,53,50,-3
9,Bergamo,Italy,1.01,59,62,3


In [151]:
df = duckdb.sql("SELECT * FROM city_metrics ORDER BY roi_centre DESC").df()
df.to_csv('../data/reference/results.csv', index=False)

In [152]:
df.head()

,city,country,city_slug,n_listings,med_airbnb_price_usd,med_nights,occupancy_pct,annual_revenue_usd,pct_capped,price_sqm_centre,price_sqm_out,price_utilities,roi_centre,roi_outside
0,Chicago,United States,chicago,2935,225.18,138.0,37.8,31075.0,19.1,4094.16,2696.73,193.44,10.98,17.19
1,Edinburgh,United Kingdom,edinburgh,2900,331.86,138.0,37.8,45796.0,27.4,6689.03,4680.85,371.64,9.63,14.19
2,Portland,United States,portland,2009,177.23,144.0,39.5,25521.0,22.4,3896.40,3372.32,290.44,8.86,10.40
3,Columbus,United States,columbus,1112,167.00,132.0,36.2,22044.0,26.2,3440.00,1775.50,222.91,8.77,17.92
4,Minneapolis,United States,twin-cities-msa,1763,197.37,102.0,27.9,20132.0,13.0,3395.59,2174.56,158.11,8.22,13.40


In [153]:
duckdb.sql("SELECT * FROM city_metrics ORDER BY roi_centre DESC").df().to_csv('../data/reference/results.csv', index=False)


In [154]:
duckdb.sql("""
SELECT ROUND(MEDIAN(availability_365), 0) AS med_avail
FROM read_csv_auto('../data/detailed/edinburgh.csv.gz')
WHERE room_type = 'Entire home/apt' AND number_of_reviews_ltm > 0 AND bedrooms <= 2
""").df()

,med_avail
0,114.0


In [156]:
duckdb.sql("""
SELECT parse_filename(filename, true) AS city,
       ROUND(MEDIAN(availability_365), 0) AS med_avail
FROM read_csv_auto('../data/detailed/*.csv.gz', filename = true, union_by_name = true)
WHERE room_type = 'Entire home/apt' AND number_of_reviews_ltm > 0 AND bedrooms <= 2
GROUP BY 1 ORDER BY med_avail
LIMIT 50
""").df()

,city,med_avail
0,copenhagen.csv,29.0
1,amsterdam.csv,36.0
2,oslo.csv,85.0
3,rotterdam.csv,94.0
4,dublin.csv,110.0
5,edinburgh.csv,114.0
6,munich.csv,114.0
7,vancouver.csv,146.0
8,paris.csv,147.0
9,sydney.csv,155.0


In [157]:
duckdb.sql("""
WITH a AS (
  SELECT parse_filename(filename, true) AS city,
         MEDIAN(availability_365) AS med_avail,
         MEDIAN(LEAST((number_of_reviews_ltm/0.5)*GREATEST(3,minimum_nights),255)) AS med_nights
  FROM read_csv_auto('../data/detailed/*.csv.gz', filename = true, union_by_name = true)
  WHERE room_type = 'Entire home/apt' AND number_of_reviews_ltm > 0 AND bedrooms <= 2
  GROUP BY 1
)
SELECT COUNT(*) AS n_cities,
       SUM(CASE WHEN med_nights > med_avail THEN 1 ELSE 0 END) AS n_exceeding
FROM a
""").df()

,n_cities,n_exceeding
0,84,2.0


In [2]:
import duckdb
duckdb.sql("SELECT city, country, roi_centre FROM city_metrics WHERE roi_centre < 0 ORDER BY roi_centre").df()

CatalogException: Catalog Error: Table with name city_metrics does not exist!
Did you mean "sqlite_master"?

In [5]:
import duckdb
duckdb.sql("SELECT city, country, roi_centre FROM read_csv_auto('../data/reference/results.csv') WHERE roi_centre < 0 ORDER BY roi_centre").df()

,city,country,roi_centre
0,Oslo,Norway,-0.74
1,Hong Kong,Hong Kong (China),-0.70
2,Munich,Germany,-0.49
3,Nairobi,Kenya,-0.26
4,London,United Kingdom,-0.26
5,Copenhagen,Denmark,-0.16
6,Amsterdam,Netherlands,-0.07
7,Bangkok,Thailand,-0.07
8,Vienna,Austria,-0.07


In [9]:
import duckdb

# Build the cities table: cities with 500+ active entire homes, joined to Numbeo property prices
duckdb.sql("""
CREATE OR REPLACE TABLE cities AS
WITH my_cities AS (
    SELECT parse_filename(filename, true) AS city,
           COUNT(*) AS active_entire_homes
    FROM read_csv_auto('../data/*.csv', filename = true, union_by_name = true)
    WHERE parse_filename(filename, true) NOT IN ('geneva','zurich','vaud')
      AND number_of_reviews_ltm > 0
      AND room_type = 'Entire home/apt'
    GROUP BY parse_filename(filename, true)
    HAVING COUNT(*) >= 500
),
numbeo AS (
    SELECT city AS numbeo_city,
           replace(lower(split_part(city, ',', 1)), ' ', '-') AS join_key,
           price_sqm
    FROM read_csv_auto('../data/reference/numbeo_centre.csv')
),
city_lookup AS (
    SELECT * FROM read_csv_auto('../data/reference/city_lookup.csv')
)
SELECT m.city, m.active_entire_homes, n.numbeo_city, n.price_sqm
FROM my_cities m
LEFT JOIN city_lookup c ON m.city = c.city
LEFT JOIN numbeo n ON n.numbeo_city = c.numbeo_city
                   OR (c.numbeo_city IS NULL AND n.join_key = m.city)
""")

duckdb.sql("DELETE FROM cities WHERE price_sqm IS NULL")

# Build city_metrics: the final results table, 82 cities
duckdb.sql("""
CREATE OR REPLACE TABLE city_metrics AS
WITH listing_nights AS (
    SELECT
        parse_filename(filename, true) AS city,
        CAST(replace(replace(price, '$', ''), ',', '') AS DOUBLE) AS price_num,
        LEAST((number_of_reviews_ltm / 0.5) * GREATEST(3, minimum_nights), 255) AS nights_booked,
        (number_of_reviews_ltm / 0.5) * GREATEST(3, minimum_nights) AS nights_uncapped
    FROM read_csv_auto('../data/detailed/*.csv.gz', filename = true, union_by_name = true)
    WHERE room_type = 'Entire home/apt'
      AND number_of_reviews_ltm > 0
      AND bedrooms <= 2
),
agg AS (
    SELECT
        replace(city, '.csv', '') AS city,
        MEDIAN(price_num) AS med_price_local,
        MEDIAN(nights_booked) AS med_nights,
        ROUND(100.0 * SUM(CASE WHEN nights_uncapped > 255 THEN 1 ELSE 0 END) / COUNT(*), 1) AS pct_capped,
        COUNT(*) AS n_listings
    FROM listing_nights
    GROUP BY 1
    HAVING COUNT(*) >= 500
),
cur AS (SELECT * FROM read_csv_auto('../data/reference/country_currency.csv')),
ex  AS (SELECT * FROM read_csv_auto('../data/reference/exchange_rates.csv')),
out AS (SELECT city AS numbeo_city, price_sqm AS price_sqm_out
        FROM read_csv_auto('../data/reference/numbeo_outside.csv')),
utl AS (SELECT city AS numbeo_city, price_utilities
        FROM read_csv_auto('../data/reference/numbeo_utilities.csv'))
SELECT
    split_part(c.numbeo_city, ',', 1) AS city,
    trim(split_part(c.numbeo_city, ',', -1)) AS country,
    c.city AS city_slug,
    a.n_listings,
    ROUND(a.med_price_local / ex.usd_rate, 2) AS med_airbnb_price_usd,
    ROUND(a.med_nights, 0) AS med_nights,
    ROUND(100.0 * a.med_nights / 365, 1) AS occupancy_pct,
    ROUND(a.med_price_local / ex.usd_rate * a.med_nights, 0) AS annual_revenue_usd,
    a.pct_capped,
    c.price_sqm AS price_sqm_centre,
    out.price_sqm_out,
    utl.price_utilities,
    ROUND(100.0 * (a.med_price_local / ex.usd_rate * a.med_nights) / (c.price_sqm * 60)
          - 100.0 * (12 * utl.price_utilities) / (85 * c.price_sqm) - 1.0, 2) AS roi_centre,
    ROUND(100.0 * (a.med_price_local / ex.usd_rate * a.med_nights) / (out.price_sqm_out * 60)
          - 100.0 * (12 * utl.price_utilities) / (85 * out.price_sqm_out) - 1.0, 2) AS roi_outside
FROM cities c
JOIN agg a   ON a.city = c.city
JOIN cur     ON cur.country = trim(split_part(c.numbeo_city, ',', -1))
JOIN ex      ON ex.currency = cur.currency
LEFT JOIN out ON out.numbeo_city = c.numbeo_city
LEFT JOIN utl ON utl.numbeo_city = c.numbeo_city
WHERE c.city <> 'new-york-city'
""")

In [10]:
duckdb.sql("SHOW TABLES").df()

,name
0,cities
1,city_metrics


In [11]:
duckdb.sql("SELECT COUNT(*) FROM city_metrics").df()

,count_star()
0,82


In [12]:
# 1. How many cities are actually negative? README says six.
duckdb.sql("SELECT city, country, roi_centre, roi_outside FROM city_metrics WHERE roi_centre < 0 ORDER BY roi_centre").df()

,city,country,roi_centre,roi_outside
0,Oslo,Norway,-0.74,-0.56
1,Hong Kong,Hong Kong (China),-0.70,-0.51
2,Munich,Germany,-0.49,-0.26
3,London,United Kingdom,-0.26,0.75
4,Nairobi,Kenya,-0.26,0.17
5,Copenhagen,Denmark,-0.16,0.17
6,Amsterdam,Netherlands,-0.07,0.44
7,Vienna,Austria,-0.07,1.22
8,Bangkok,Thailand,-0.07,0.73


In [13]:
# 2. Which cities have identical centre/outside prices? (Rochester was dropped)
duckdb.sql("SELECT city, country, price_sqm_centre, price_sqm_out FROM city_metrics WHERE price_sqm_centre = price_sqm_out").df()

,city,country,price_sqm_centre,price_sqm_out


In [14]:
# 3. Does outlier trimming change the medians? The README claims it doesn't.
duckdb.sql("""
WITH p AS (
  SELECT replace(parse_filename(filename,true),'.csv','') AS city,
         CAST(replace(replace(price,'$',''),',','') AS DOUBLE) AS pr
  FROM read_csv_auto('../data/detailed/*.csv.gz', filename=true, union_by_name=true)
  WHERE room_type='Entire home/apt' AND number_of_reviews_ltm>0 AND bedrooms<=2
),
b AS (
  SELECT city,
         quantile_cont(pr, 0.01) AS lo,
         quantile_cont(pr, 0.99) AS hi
  FROM p GROUP BY city
)
SELECT p.city,
       ROUND(MEDIAN(p.pr), 2) AS med_all,
       ROUND(MEDIAN(CASE WHEN p.pr BETWEEN b.lo AND b.hi THEN p.pr END), 2) AS med_trimmed
FROM p JOIN b ON b.city = p.city
GROUP BY p.city
ORDER BY ABS(med_all - med_trimmed) DESC
LIMIT 10
""").df()

,city,med_all,med_trimmed
0,bogota,174553.00,174500.00
1,nairobi,5985.00,5965.67
2,budapest,30877.75,30872.50
3,santiago,62463.00,62462.00
4,new-york-city,200.97,200.07
5,athens,99.00,99.01
6,antwerp,141.00,141.00
7,bergamo,125.25,125.25
8,winnipeg,137.00,137.00
9,portland,177.23,177.23


In [15]:
import numpy as np
d = duckdb.sql("SELECT * FROM city_metrics").df()

print("revenue vs price:", d["annual_revenue_usd"].corr(d["price_sqm_centre"]))
print("log price vs roi:", np.log(d["price_sqm_centre"]).corr(d["roi_centre"]))
print("1/price vs roi:", (1/d["price_sqm_centre"]).corr(d["roi_centre"]))
print("rev/price vs roi:", (d["annual_revenue_usd"]/d["price_sqm_centre"]).corr(d["roi_centre"]))

revenue vs price: 0.07648171371380373
log price vs roi: -0.27272816413342554
1/price vs roi: 0.12297889092125404
rev/price vs roi: 0.9956865133743671


In [16]:
import numpy as np
print("log rev vs log roi:  ", np.log(d["annual_revenue_usd"]).corr(np.log(d["roi_centre"].clip(lower=0.01))))
print("log price vs log roi:", np.log(d["price_sqm_centre"]).corr(np.log(d["roi_centre"].clip(lower=0.01))))
print("sd log revenue:", np.log(d["annual_revenue_usd"]).std())
print("sd log price:  ", np.log(d["price_sqm_centre"]).std())

log rev vs log roi:   0.6726849552826834
log price vs log roi: -0.36474834757060104
sd log revenue: 0.6707272948746511
sd log price:   0.5278437409064908


In [17]:
duckdb.sql("""
SELECT replace(parse_filename(filename,true),'.csv','') AS city,
       ROUND(quantile_cont(pr,0.99),0) AS p99,
       ROUND(MAX(pr),0) AS max_price,
       SUM(CASE WHEN pr > quantile_cont(pr,0.99) OVER () THEN 1 ELSE 0 END) AS n_above
FROM (SELECT filename, CAST(replace(replace(price,'$',''),',','') AS DOUBLE) AS pr
      FROM read_csv_auto('../data/detailed/london.csv.gz', filename=true))
GROUP BY 1
""").df()

BinderException: Binder Error: aggregate function calls cannot contain window function calls

In [18]:
duckdb.sql("""
SELECT COUNT(*) FROM cities c
JOIN (SELECT replace(parse_filename(filename,true),'.csv','') AS city, COUNT(*) n
      FROM read_csv_auto('../data/detailed/*.csv.gz', filename=true, union_by_name=true)
      WHERE room_type='Entire home/apt' AND number_of_reviews_ltm>0 AND bedrooms<=2
      GROUP BY 1 HAVING COUNT(*)>=500) a ON a.city=c.city
WHERE c.city <> 'new-york-city'
""").df()

,count_star()
0,82


In [19]:
duckdb.sql("""
SELECT ROUND(MEDIAN(pr),0) AS median,
       ROUND(quantile_cont(pr,0.95),0) AS p95,
       ROUND(quantile_cont(pr,0.99),0) AS p99,
       ROUND(MAX(pr),0) AS max_price,
       SUM(CASE WHEN pr > 5000 THEN 1 ELSE 0 END) AS n_over_5000
FROM (SELECT CAST(replace(replace(price,'$',''),',','') AS DOUBLE) AS pr
      FROM read_csv_auto('../data/detailed/london.csv.gz')
      WHERE room_type='Entire home/apt' AND number_of_reviews_ltm>0 AND bedrooms<=2)
""").df()

,median,p95,p99,max_price,n_over_5000
0,219.0,536.0,843.0,10071.0,7.0


In [20]:
duckdb.sql("""
SELECT
    COUNT(*) AS cities_strict,
    SUM(CASE WHEN n_with_fallback >= 500 AND n_strict < 500 THEN 1 ELSE 0 END) AS would_be_added
FROM (
    SELECT replace(parse_filename(filename,true),'.csv','') AS city,
        SUM(CASE WHEN bedrooms <= 2 THEN 1 ELSE 0 END) AS n_strict,
        SUM(CASE WHEN bedrooms <= 2 OR (bedrooms IS NULL AND accommodates <= 4) THEN 1 ELSE 0 END) AS n_with_fallback
    FROM read_csv_auto('../data/detailed/*.csv.gz', filename=true, union_by_name=true)
    WHERE room_type='Entire home/apt' AND number_of_reviews_ltm>0
    GROUP BY 1
)
""").df()

,cities_strict,would_be_added
0,84,0.0


In [22]:
duckdb.sql("""
SELECT replace(parse_filename(filename,true),'.csv','') AS city,
    ROUND(MEDIAN(pr),0) AS median,
    ROUND(quantile_cont(pr,0.99),0) AS p99,
    ROUND(MAX(pr),0) AS max_price,
    ROUND(MAX(pr)/quantile_cont(pr,0.99),1) AS max_over_p99
FROM (SELECT filename, CAST(replace(replace(price,'$',''),',','') AS DOUBLE) AS pr
      FROM read_csv_auto('../data/detailed/*.csv.gz', filename=true, union_by_name=true)
      WHERE room_type='Entire home/apt' AND number_of_reviews_ltm>0 AND bedrooms<=2)
GROUP BY 1 ORDER BY max_over_p99 DESC LIMIT 50
""").df()

,city,median,p99,max_price,max_over_p99
0,buenos-aires,103226.0,443003.0,171643781.0,387.5
1,bangkok,1619.0,6680.0,1157979.0,173.3
2,bogota,174553.0,1083502.0,174600081.0,161.1
3,prague,2519.0,8345.0,1191906.0,142.8
4,santiago,62463.0,445796.0,43741160.0,98.1
5,san-diego,303.0,1089.0,82341.0,75.6
6,sao-paulo,339.0,1107.0,57127.0,51.6
7,budapest,30878.0,160315.0,8258389.0,51.5
8,rio-de-janeiro,422.0,2240.0,114118.0,50.9
9,milan,137.0,675.0,28587.0,42.4


In [23]:
import duckdb

# Build the cities table: cities with 500+ active entire homes, joined to Numbeo property prices
duckdb.sql("""
CREATE OR REPLACE TABLE cities AS
WITH my_cities AS (
    SELECT parse_filename(filename, true) AS city,
           COUNT(*) AS active_entire_homes
    FROM read_csv_auto('../data/*.csv', filename = true, union_by_name = true)
    WHERE parse_filename(filename, true) NOT IN ('geneva','zurich','vaud')
      AND number_of_reviews_ltm > 0
      AND room_type = 'Entire home/apt'
    GROUP BY parse_filename(filename, true)
    HAVING COUNT(*) >= 500
),
numbeo AS (
    SELECT city AS numbeo_city,
           replace(lower(split_part(city, ',', 1)), ' ', '-') AS join_key,
           price_sqm
    FROM read_csv_auto('../data/reference/numbeo_centre.csv')
),
city_lookup AS (
    SELECT * FROM read_csv_auto('../data/reference/city_lookup.csv')
)
SELECT m.city, m.active_entire_homes, n.numbeo_city, n.price_sqm
FROM my_cities m
LEFT JOIN city_lookup c ON m.city = c.city
LEFT JOIN numbeo n ON n.numbeo_city = c.numbeo_city
                   OR (c.numbeo_city IS NULL AND n.join_key = m.city)
""")

duckdb.sql("DELETE FROM cities WHERE price_sqm IS NULL")

# Build city_metrics: the final results table, 82 cities, prices trimmed at the 1st and 99th percentile
duckdb.sql("""
CREATE OR REPLACE TABLE city_metrics AS
WITH raw AS (
    SELECT
        replace(parse_filename(filename, true), '.csv', '') AS city,
        CAST(replace(replace(price, '$', ''), ',', '') AS DOUBLE) AS price_num,
        number_of_reviews_ltm,
        minimum_nights
    FROM read_csv_auto('../data/detailed/*.csv.gz', filename = true, union_by_name = true)
    WHERE room_type = 'Entire home/apt'
      AND number_of_reviews_ltm > 0
      AND bedrooms <= 2
      AND price IS NOT NULL
),
bounds AS (
    SELECT city,
           quantile_cont(price_num, 0.01) AS lo,
           quantile_cont(price_num, 0.99) AS hi
    FROM raw
    GROUP BY city
),
listing_nights AS (
    SELECT
        r.city,
        r.price_num,
        LEAST((r.number_of_reviews_ltm / 0.5) * GREATEST(3, r.minimum_nights), 255) AS nights_booked,
        (r.number_of_reviews_ltm / 0.5) * GREATEST(3, r.minimum_nights) AS nights_uncapped
    FROM raw r
    JOIN bounds b ON b.city = r.city
    WHERE r.price_num BETWEEN b.lo AND b.hi
),
agg AS (
    SELECT
        city,
        MEDIAN(price_num) AS med_price_local,
        MEDIAN(nights_booked) AS med_nights,
        ROUND(100.0 * SUM(CASE WHEN nights_uncapped > 255 THEN 1 ELSE 0 END) / COUNT(*), 1) AS pct_capped,
        COUNT(*) AS n_listings
    FROM listing_nights
    GROUP BY 1
    HAVING COUNT(*) >= 500
),
cur AS (SELECT * FROM read_csv_auto('../data/reference/country_currency.csv')),
ex  AS (SELECT * FROM read_csv_auto('../data/reference/exchange_rates.csv')),
out AS (SELECT city AS numbeo_city, price_sqm AS price_sqm_out
        FROM read_csv_auto('../data/reference/numbeo_outside.csv')),
utl AS (SELECT city AS numbeo_city, price_utilities
        FROM read_csv_auto('../data/reference/numbeo_utilities.csv'))
SELECT
    split_part(c.numbeo_city, ',', 1) AS city,
    trim(split_part(c.numbeo_city, ',', -1)) AS country,
    c.city AS city_slug,
    a.n_listings,
    ROUND(a.med_price_local / ex.usd_rate, 2) AS med_airbnb_price_usd,
    ROUND(a.med_nights, 0) AS med_nights,
    ROUND(100.0 * a.med_nights / 365, 1) AS occupancy_pct,
    ROUND(a.med_price_local / ex.usd_rate * a.med_nights, 0) AS annual_revenue_usd,
    a.pct_capped,
    c.price_sqm AS price_sqm_centre,
    out.price_sqm_out,
    utl.price_utilities,
    ROUND(100.0 * (a.med_price_local / ex.usd_rate * a.med_nights) / (c.price_sqm * 60)
          - 100.0 * (12 * utl.price_utilities) / (85 * c.price_sqm) - 1.0, 2) AS roi_centre,
    ROUND(100.0 * (a.med_price_local / ex.usd_rate * a.med_nights) / (out.price_sqm_out * 60)
          - 100.0 * (12 * utl.price_utilities) / (85 * out.price_sqm_out) - 1.0, 2) AS roi_outside
FROM cities c
JOIN agg a   ON a.city = c.city
JOIN cur     ON cur.country = trim(split_part(c.numbeo_city, ',', -1))
JOIN ex      ON ex.currency = cur.currency
LEFT JOIN out ON out.numbeo_city = c.numbeo_city
LEFT JOIN utl ON utl.numbeo_city = c.numbeo_city
WHERE c.city <> 'new-york-city'
""")

In [25]:
duckdb.sql("SELECT city, country, roi_centre, roi_outside FROM city_metrics ORDER BY roi_centre DESC LIMIT 50").df()

,city,country,roi_centre,roi_outside
0,Chicago,United States,11.53,18.03
1,Edinburgh,United Kingdom,10.12,14.89
2,Portland,United States,9.77,11.45
3,Columbus,United States,9.74,19.80
4,Minneapolis,United States,8.22,13.40
5,Fort Worth,United States,8.12,19.52
6,Victoria,Canada,7.39,9.11
7,Barcelona,Spain,6.71,10.22
8,Winnipeg,Canada,6.61,6.60
9,San Diego,United States,6.45,8.17


In [26]:
duckdb.sql("SELECT * FROM city_metrics ORDER BY roi_centre DESC").df().to_csv('../data/reference/results.csv', index=False)

In [27]:
duckdb.sql("""
WITH raw AS (
    SELECT replace(parse_filename(filename,true),'.csv','') AS city,
           CAST(replace(replace(price,'$',''),',','') AS DOUBLE) AS pr,
           LEAST((number_of_reviews_ltm/0.5)*GREATEST(3,minimum_nights),255) AS nights
    FROM read_csv_auto('../data/detailed/*.csv.gz', filename=true, union_by_name=true)
    WHERE room_type='Entire home/apt' AND number_of_reviews_ltm>0 AND bedrooms<=2 AND price IS NOT NULL
),
b AS (SELECT city, quantile_cont(pr,0.01) lo, quantile_cont(pr,0.99) hi FROM raw GROUP BY 1)
SELECT r.city,
    ROUND(MEDIAN(r.pr),1) AS price_all,
    ROUND(MEDIAN(CASE WHEN r.pr BETWEEN b.lo AND b.hi THEN r.pr END),1) AS price_trim,
    ROUND(MEDIAN(r.nights),0) AS nights_all,
    ROUND(MEDIAN(CASE WHEN r.pr BETWEEN b.lo AND b.hi THEN r.nights END),0) AS nights_trim
FROM raw r JOIN b ON b.city=r.city
WHERE r.city IN ('chicago','portland','columbus','edinburgh')
GROUP BY 1
""").df()

,city,price_all,price_trim,nights_all,nights_trim
0,portland,177.2,177.2,156.0,156.0
1,columbus,167.0,167.0,138.0,144.0
2,chicago,225.2,225.2,144.0,144.0
3,edinburgh,251.0,251.0,144.0,144.0


In [28]:
duckdb.sql("""
WITH raw AS (
    SELECT replace(parse_filename(filename,true),'.csv','') AS city,
           price,
           LEAST((number_of_reviews_ltm/0.5)*GREATEST(3,minimum_nights),255) AS nights
    FROM read_csv_auto('../data/detailed/*.csv.gz', filename=true, union_by_name=true)
    WHERE room_type='Entire home/apt' AND number_of_reviews_ltm>0 AND bedrooms<=2
)
SELECT city,
    ROUND(MEDIAN(nights),0) AS nights_all_listings,
    ROUND(MEDIAN(CASE WHEN price IS NOT NULL THEN nights END),0) AS nights_priced_only
FROM raw
WHERE city IN ('chicago','portland','columbus','edinburgh','london','paris')
GROUP BY 1
""").df()

,city,nights_all_listings,nights_priced_only
0,portland,144.0,156.0
1,chicago,138.0,144.0
2,paris,60.0,60.0
3,edinburgh,138.0,144.0
4,london,42.0,42.0
5,columbus,132.0,138.0


In [29]:
# 1. export + top of ranking
d = duckdb.sql("SELECT * FROM city_metrics ORDER BY roi_centre DESC").df()
d.to_csv('../data/reference/results.csv', index=False)
print(len(d))
d[["city","country","roi_centre","roi_outside","occupancy_pct","annual_revenue_usd"]].head(12)

80


,city,country,roi_centre,roi_outside,occupancy_pct,annual_revenue_usd
0,Chicago,United States,11.53,18.03,39.5,32426.0
1,Edinburgh,United Kingdom,10.12,14.89,39.5,47787.0
2,Portland,United States,9.77,11.45,42.7,27648.0
3,Columbus,United States,9.74,19.80,39.5,24048.0
4,Minneapolis,United States,8.22,13.40,27.9,20132.0
5,Fort Worth,United States,8.12,19.52,34.5,24255.0
6,Victoria,Canada,7.39,9.11,41.1,28137.0
7,Barcelona,Spain,6.71,10.22,41.1,36817.0
8,Winnipeg,Canada,6.61,6.60,34.5,12126.0
9,San Diego,United States,6.45,8.17,34.5,38225.0


In [30]:
# 2. negative cities + Boston
duckdb.sql("""
SELECT city, country, roi_centre, roi_outside, occupancy_pct, annual_revenue_usd
FROM city_metrics WHERE roi_centre < 0 OR city = 'Boston' ORDER BY roi_centre
""").df()

,city,country,roi_centre,roi_outside,occupancy_pct,annual_revenue_usd
0,Oslo,Norway,-0.74,-0.56,8.2,5434.0
1,Hong Kong,Hong Kong (China),-0.68,-0.48,15.3,7370.0
2,Munich,Germany,-0.49,-0.26,9.9,7260.0
3,Nairobi,Kenya,-0.27,0.17,6.6,1106.0
4,London,United Kingdom,-0.26,0.75,11.5,12161.0
5,Copenhagen,Denmark,-0.16,0.17,8.2,7489.0
6,Bangkok,Thailand,-0.12,0.63,23.0,4094.0
7,Vienna,Austria,-0.07,1.22,21.4,10986.0
8,Boston,United States,3.93,9.44,36.2,42680.0


In [31]:
# 3. correlations + spread by country
print(d["annual_revenue_usd"].corr(d["roi_centre"]),
      d["price_sqm_centre"].corr(d["roi_centre"]),
      d["annual_revenue_usd"].corr(d["price_sqm_centre"]))
d["spread"] = d["roi_outside"] - d["roi_centre"]
g = d.groupby("country").agg(n=("city","count"), mean_spread=("spread","mean"))
print(g[g["n"] >= 3].sort_values("mean_spread", ascending=False).round(2))

0.7715156619641029 -0.36116345612902995 0.0735934163250448
                 n  mean_spread
country                        
United States   21         5.10
Spain            5         3.75
Belgium          3         2.73
Italy            6         2.31
Canada           7         1.83
United Kingdom   4         1.71
Australia        3         1.03
France           3         0.90


In [32]:
duckdb.sql("SELECT city FROM cities WHERE city NOT IN (SELECT city_slug FROM city_metrics) AND city <> 'new-york-city'").df()

,city
0,rotterdam
1,the-hague
2,rochester


In [33]:
print("capped vs occupancy:", d["pct_capped"].corr(d["occupancy_pct"]))
print("nightly vs property price:", d["med_airbnb_price_usd"].corr(d["price_sqm_centre"]))
print("listings vs roi:", d["n_listings"].corr(d["roi_centre"]))

capped vs occupancy: 0.8774255336225322
nightly vs property price: 0.4090684963426397
listings vs roi: -0.3717192924606052


In [34]:
import numpy as np
lr = np.log(d["annual_revenue_usd"]); lm = np.log(d["price_sqm_centre"])
print("var log revenue:", round(lr.var(),4))
print("var log price:  ", round(lm.var(),4))
print("covariance:     ", round(np.cov(lr,lm)[0,1],4))
print("revenue share of ROI variation:", round(lr.var()/(lr.var()+lm.var()),3))

var log revenue: 0.4607
var log price:   0.2857
covariance:      0.13
revenue share of ROI variation: 0.617


In [35]:
print("nightly vs utilities: ", round(d["med_airbnb_price_usd"].corr(d["price_utilities"]),3))
print("property vs utilities:", round(d["price_sqm_centre"].corr(d["price_utilities"]),3))

nightly vs utilities:  0.437
property vs utilities: 0.385


In [36]:
print("raw:", round(d["annual_revenue_usd"].corr(d["price_sqm_centre"]),3))
print("log:", round(np.log(d["annual_revenue_usd"]).corr(np.log(d["price_sqm_centre"])),3))

raw: 0.074
log: 0.358


In [37]:
# 1. spreads
for c in ["med_airbnb_price_usd","occupancy_pct","price_sqm_centre"]:
    print(c, round(d[c].max()/d[c].min(),1))

# 2. does any city's median approach the cap?
print("max occupancy:", d["occupancy_pct"].max(), "| max pct_capped:", d["pct_capped"].max())

# 3. the ROI crowding around Riga
print(d.sort_values("roi_centre", ascending=False)[["city","roi_centre"]].iloc[60:78].to_string())

med_airbnb_price_usd 7.8
occupancy_pct 7.1
price_sqm_centre 19.6
max occupancy: 46.8 | max pct_capped: 29.1
              city  roi_centre
60  Rio de Janeiro        0.97
61          Sydney        0.96
62        Bordeaux        0.62
63          Bogota        0.38
64           Paris        0.33
65            Riga        0.29
66           Tokyo        0.27
67       Amsterdam        0.27
68       Stockholm        0.26
69    Thessaloniki        0.12
70          Taipei        0.11
71           Milan        0.09
72          Vienna       -0.07
73         Bangkok       -0.12
74      Copenhagen       -0.16
75          London       -0.26
76         Nairobi       -0.27
77          Munich       -0.49


In [40]:
duckdb.sql("""
SELECT
    replace(parse_filename(filename,true),'.csv','') AS city,
    ROUND(AVG(CASE WHEN price IS NOT NULL THEN availability_365 END),0) AS avail_priced,
    ROUND(AVG(CASE WHEN price IS NULL     THEN availability_365 END),0) AS avail_null,
    ROUND(AVG(CASE WHEN price IS NOT NULL THEN number_of_reviews_ltm END),1) AS rev_priced,
    ROUND(AVG(CASE WHEN price IS NULL     THEN number_of_reviews_ltm END),1) AS rev_null,
    SUM(CASE WHEN price IS NULL THEN 1 ELSE 0 END) AS n_null
FROM read_csv_auto('../data/detailed/*.csv.gz', filename=true, union_by_name=true)
WHERE room_type='Entire home/apt' AND number_of_reviews_ltm > 0 AND bedrooms <= 2
GROUP BY 1
HAVING SUM(CASE WHEN price IS NULL THEN 1 ELSE 0 END) > 0
ORDER BY rev_null DESC
LIMIT 50
""").df()

,city,avail_priced,avail_null,rev_priced,rev_null,n_null
0,quebec-city,229.0,172.0,26.2,27.1,15.0
1,porto,259.0,232.0,19.5,20.5,535.0
2,athens,257.0,250.0,18.8,19.2,20.0
3,edinburgh,148.0,44.0,30.1,17.6,268.0
4,bogota,305.0,290.0,15.9,16.9,45.0
5,antwerp,215.0,153.0,20.8,16.2,133.0
6,winnipeg,260.0,59.0,24.0,16.0,23.0
7,budapest,184.0,30.0,26.4,14.7,462.0
8,mexico-city,255.0,50.0,23.4,14.6,308.0
9,brussels,202.0,38.0,25.8,14.3,250.0


In [41]:
d["util_share"] = (12*d["price_utilities"])/(85*d["price_sqm_centre"])*100
print(d.nlargest(10,"util_share")[["city","roi_centre","util_share"]].to_string())

            city  roi_centre  util_share
65          Riga        0.29    1.659715
57  Buenos Aires        1.13    1.103035
2       Portland        9.77    1.052338
8       Winnipeg        6.61    1.019931
3       Columbus        9.74    0.914815
41    Manchester        2.23    0.853687
69  Thessaloniki        0.12    0.843911
59       Bergamo        1.01    0.821002
51        Athens        1.55    0.797636
52      Santiago        1.54    0.789110


In [42]:
df = duckdb.sql("SELECT * FROM city_metrics").df()
(df['price_sqm_centre']*60).corr(df['roi_centre'])

np.float64(-0.3611634561290303)